In [10]:
%cd /mnt/models/huggingface/apple-depth-pro
import sys
sys.path.append('/mnt/models/huggingface/apple-depth-pro/src')
import depth_pro

import numpy as np

import torch
import cv_depot.api as cvd

In [ ]:
device = torch.device('cuda')
model = get_depth_model(device)

In [55]:
def get_depth_model(device):
    model, _ = depth_pro.create_model_and_transforms(
        device=device,
        precision=torch.float16,
    )
    model.eval()
    return model

def create_depth_mask(image, model, device):
    img = image.to_bit_depth(cvd.BitDepth.FLOAT16)
    array = np.transpose(img.data, (2, 0, 1))
    tensor = torch.from_numpy(array).to(device)
    depth = model.infer(tensor)['depth'].cpu().numpy()
    depth = cvd.Image.from_array(depth)
    depth = cvd.ops.channel.invert(depth)
    
    cmap = cvd.ChannelMap({'r': '0.r', 'g': '0.g', 'b': '0.b', 'depth.z': '1.l'})
    return cvd.ops.channel.remap([img, depth], cmap)

In [57]:
src = '/mnt/models/huggingface/apple-depth-pro/data/example.jpg'
img = cvd.Image.read(src)
img = create_depth_mask(img, model, device)
tgt = '/mnt/storage/data/example.exr'
img.write(tgt)

In [52]:
model

DepthPro(
  (encoder): DepthProEncoder(
    (patch_encoder): VisionTransformer(
      (patch_embed): PatchEmbed(
        (proj): Conv2d(3, 1024, kernel_size=(16, 16), stride=(16, 16))
        (norm): Identity()
      )
      (pos_drop): Dropout(p=0.0, inplace=False)
      (patch_drop): Identity()
      (norm_pre): Identity()
      (blocks): Sequential(
        (0): Block(
          (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
          (attn): Attention(
            (qkv): Linear(in_features=1024, out_features=3072, bias=True)
            (q_norm): Identity()
            (k_norm): Identity()
            (attn_drop): Dropout(p=0.0, inplace=False)
            (proj): Linear(in_features=1024, out_features=1024, bias=True)
            (proj_drop): Dropout(p=0.0, inplace=False)
          )
          (ls1): LayerScale()
          (drop_path1): Identity()
          (norm2): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
          (mlp): Mlp(
            (fc1): Linea